<a href="https://colab.research.google.com/github/Adityaraj1969/india-runs-candidate-ranking/blob/main/india_runs_candidate_ranking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏆 India Runs Hackathon — Intelligent Candidate Ranking
**Track 1: Data & AI Challenge by Redrob**

**Job:** Senior AI Engineer — Founding Team at Redrob AI  
**Task:** Rank 100,000 candidates. Output top 100 with scores + reasoning.

---
### Our approach (5 signals combined):
1. **Semantic similarity** — AI understands meaning, not just keywords (45%)
2. **Skill match** — required AI/ML skills explicitly matched (25%)
3. **Role relevance** — is their job title/career in AI/engineering? (15%)
4. **Behavioral availability** — are they actually reachable and active? (10%)
5. **Profile quality** — completeness, GitHub, assessments (5%)

**Key insight:** We use behavioral signals as a MULTIPLIER to penalize
perfect-on-paper candidates who are unreachable or inactive.

## CELL 1 — Install libraries
Run this first. It downloads the AI model (~90MB). Takes 2-3 minutes.

In [ ]:
# Install all required libraries
# The ! means 'run this as a terminal command'
!pip install sentence-transformers scikit-learn pandas numpy tqdm --quiet

print('✅ All libraries installed!')

✅ All libraries installed!


## CELL 2 — Connect Google Drive
Your dataset must be uploaded to Google Drive first.
Upload the hackathon files into a folder called **hackathon_data** in your Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ⚠️ CHANGE THIS if your folder name is different
DATASET_FOLDER = '/content/drive/MyDrive/hackathon_data'

print('Files in your dataset folder:')
for f in os.listdir(DATASET_FOLDER):
    size_mb = os.path.getsize(os.path.join(DATASET_FOLDER, f)) / 1024 / 1024
    print(f'  {f}  ({size_mb:.1f} MB)')

Mounted at /content/drive
Files in your dataset folder:
  candidate_schema.json  (0.0 MB)
  job_description.docx  (0.0 MB)
  validate_submission.py  (0.0 MB)
  submission_metadata_template.yaml  (0.0 MB)
  sample_candidates.json  (0.3 MB)
  candidates.jsonl  (464.7 MB)
  sample_submission.csv  (0.0 MB)


## CELL 3 — Import everything we need
Run once. No output expected (except the success message).

In [ ]:
import json
import pandas as pd
import numpy as np
from datetime import date, datetime
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm   # shows a progress bar

print('✅ All imports successful!')

✅ All imports successful!


## CELL 4 — Define the job description
This is what we're ranking candidates AGAINST.
We write the key requirements as a single text block for the AI to understand.

In [ ]:
# The job description — condensed to the key signals
# We include both explicit skills AND the spirit of what they want
JOB_DESCRIPTION = """
Senior AI Engineer at Redrob AI, a Series A AI-native talent intelligence platform.
Location: Pune or Noida, India. Hybrid work mode.
Experience: 5 to 9 years in applied machine learning and AI engineering.

Must have: Production experience with embeddings-based retrieval systems using
sentence-transformers, OpenAI embeddings, BGE, or E5 models. Experience with
vector databases such as Pinecone, Weaviate, Qdrant, Milvus, FAISS, or Elasticsearch.
Strong Python programming skills. Hands-on experience designing evaluation frameworks
for ranking systems including NDCG, MRR, MAP metrics. Hybrid search infrastructure.

Nice to have: LLM fine-tuning with LoRA, QLoRA, PEFT. Learning to rank models
using XGBoost or neural methods. NLP and information retrieval. Recommendation systems.
Distributed systems and large-scale ML inference. Open source AI contributions.
RAG systems, knowledge graphs, transformer models.

Role responsibilities: Own the intelligence layer — candidate-JD matching, ranking,
and retrieval systems. Ship v2 ranking system with embeddings and hybrid retrieval.
Set up evaluation infrastructure. Mentor junior engineers. Work with product managers.

Ideal candidate: Applied ML engineer at a product company. Has shipped ranking,
search or recommendation systems to real users. Strong opinions on retrieval,
evaluation, and LLM integration. Active job seeker in the market.
"""

# The REQUIRED AI/ML skills — used for explicit keyword matching (Signal 2)
REQUIRED_SKILLS = [
    # Core AI/ML
    'embeddings', 'vector database', 'semantic search', 'sentence-transformers',
    'faiss', 'pinecone', 'weaviate', 'qdrant', 'elasticsearch', 'opensearch',
    'python', 'nlp', 'information retrieval', 'ranking', 'recommendation systems',
    # ML frameworks
    'pytorch', 'tensorflow', 'scikit-learn', 'hugging face', 'transformers',
    'langchain', 'llm', 'fine-tuning', 'rag', 'xgboost', 'lightgbm',
    # Data engineering (relevant)
    'spark', 'kafka', 'airflow', 'mlflow', 'kubeflow', 'weights & biases',
    # AI-adjacent
    'machine learning', 'deep learning', 'neural network', 'bert', 'gpt',
    'image classification', 'object detection', 'speech recognition',
    'data science', 'feature engineering', 'model deployment',
]

# Titles that strongly suggest an AI/engineering background (Signal 3)
GOOD_TITLES = [
    'ai engineer', 'ml engineer', 'machine learning', 'data scientist',
    'software engineer', 'backend engineer', 'full stack', 'data engineer',
    'nlp engineer', 'research engineer', 'research scientist', 'tech lead',
    'principal engineer', 'senior engineer', 'engineering manager',
    'platform engineer', 'infrastructure engineer', 'devops', 'mlops',
    'applied scientist', 'ai researcher',
]

# Consulting companies — JD explicitly says these are a poor fit
CONSULTING_FIRMS = [
    'tcs', 'tata consultancy', 'infosys', 'wipro', 'accenture',
    'cognizant', 'capgemini', 'hcl', 'tech mahindra', 'mphasis'
]

# Target locations — Pune/Noida preferred
PREFERRED_LOCATIONS = ['pune', 'noida', 'delhi', 'gurgaon', 'gurugram', 'hyderabad', 'mumbai', 'bangalore', 'bengaluru']

print('✅ Job description and signal lists defined!')
print(f'   Required skills to match: {len(REQUIRED_SKILLS)}')

✅ Job description and signal lists defined!
   Required skills to match: 43


## CELL 5 — Load the AI model
This downloads a pretrained AI model that converts text into numbers.
Takes ~1 minute. You'll see a progress bar.

In [ ]:
print('Loading AI model... (this takes ~1 minute)')

# all-MiniLM-L6-v2 is fast, small (90MB), and works great for this task
model = SentenceTransformer('all-MiniLM-L6-v2')

# Convert the job description into a vector (list of 384 numbers)
job_vector = model.encode([JOB_DESCRIPTION])

print(f'✅ Model loaded!')
print(f'   Job description vector shape: {job_vector.shape}')
print(f'   (384 numbers represent the meaning of the job description)')

Loading AI model... (this takes ~1 minute)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Model loaded!
   Job description vector shape: (1, 384)
   (384 numbers represent the meaning of the job description)


## CELL 6 — Helper functions (the brain of our system)
These functions score each candidate on signals 2, 3, 4, and 5.
Read the comments — every line is explained.

In [ ]:
def build_candidate_text(candidate):
    """
    Convert a candidate's entire profile into ONE text string.
    The AI model will convert this text into numbers.
    We include everything important: headline, summary, job descriptions, skills.
    """
    p = candidate['profile']
    parts = []

    # Basic profile info
    parts.append(p.get('headline', ''))
    parts.append(p.get('summary', ''))
    parts.append(f"Current role: {p.get('current_title', '')} at {p.get('current_company', '')}")
    parts.append(f"Industry: {p.get('current_industry', '')}")

    # Career history descriptions — THIS IS WHERE HIDDEN GEMS HIDE
    # e.g., someone who built vector search but doesn't list it as a skill
    for job in candidate.get('career_history', []):
        parts.append(f"{job.get('title', '')} at {job.get('company', '')}: {job.get('description', '')}")

    # Skills list
    skill_names = [s['name'] for s in candidate.get('skills', [])]
    parts.append('Skills: ' + ', '.join(skill_names))

    # Education
    for edu in candidate.get('education', []):
        parts.append(f"{edu.get('degree', '')} in {edu.get('field_of_study', '')} from {edu.get('institution', '')}")

    # Certifications
    for cert in candidate.get('certifications', []):
        parts.append(f"Certified: {cert.get('name', '')} by {cert.get('issuer', '')}")

    return ' '.join(parts)


def score_skill_match(candidate, required_skills):
    """
    Signal 2: How many required AI/ML skills does this candidate have?
    Returns a score from 0.0 to 1.0
    """
    # Get all skill names from the candidate (lowercase for comparison)
    candidate_skills = ' '.join([s['name'].lower() for s in candidate.get('skills', [])])

    # Also check career history descriptions for skill mentions
    for job in candidate.get('career_history', []):
        candidate_skills += ' ' + job.get('description', '').lower()

    # Count how many required skills appear
    matches = sum(1 for skill in required_skills if skill.lower() in candidate_skills)

    # But also weight by proficiency — expert/advanced skills count more
    proficiency_bonus = 0
    for s in candidate.get('skills', []):
        if s['name'].lower() in [r.lower() for r in required_skills]:
            if s.get('proficiency') in ('expert', 'advanced'):
                proficiency_bonus += 0.05  # small bonus for high proficiency

    # Normalize: 10+ matching skills = max score
    base_score = min(matches / 10.0, 1.0)
    return min(base_score + proficiency_bonus, 1.0)


def score_role_relevance(candidate, good_titles, consulting_firms):
    """
    Signal 3: Is this person actually working in AI/engineering?
    This catches the TRAP: candidates with AI skills but non-engineering jobs.
    Returns a score from 0.0 to 1.0
    """
    current_title = candidate['profile'].get('current_title', '').lower()
    score = 0.0

    # Check if current title matches good engineering titles
    for title in good_titles:
        if title in current_title:
            score = 0.9
            break

    # Check career history — have they EVER been in AI/engineering?
    for job in candidate.get('career_history', []):
        job_title = job.get('title', '').lower()
        for title in good_titles:
            if title in job_title:
                score = max(score, 0.5)  # past experience = partial credit
                break

    # PENALTY: Entire career only at consulting firms
    all_companies = ' '.join([job.get('company', '').lower() for job in candidate.get('career_history', [])])
    consulting_count = sum(1 for firm in consulting_firms if firm in all_companies)
    total_jobs = len(candidate.get('career_history', []))

    if total_jobs > 0 and consulting_count == total_jobs:
        # ALL jobs at consulting firms — apply penalty
        score = score * 0.5

    # Experience years check: JD wants 5-9 years
    years_exp = candidate['profile'].get('years_of_experience', 0)
    if 4 <= years_exp <= 10:
        score = min(score + 0.1, 1.0)  # sweet spot bonus
    elif years_exp < 2:
        score = score * 0.6  # too junior

    return min(score, 1.0)


def score_behavioral_availability(candidate):
    """
    Signal 4: Is this candidate actually REACHABLE and AVAILABLE?
    This is used as a MULTIPLIER — a perfect candidate who is unreachable
    still gets ranked lower. This is the key insight from the JD.
    Returns a multiplier from 0.3 to 1.0 (never 0 — they might come back)
    """
    sig = candidate.get('redrob_signals', {})
    score = 0.0

    # 1. Open to work flag — most important single signal
    if sig.get('open_to_work_flag', False):
        score += 0.30
    else:
        score += 0.05  # small score — might still be persuadable

    # 2. Last active date — recent login = actually available
    try:
        last_active = datetime.strptime(sig.get('last_active_date', '2020-01-01'), '%Y-%m-%d').date()
        days_inactive = (date.today() - last_active).days
        if days_inactive <= 30:    # active this month
            score += 0.25
        elif days_inactive <= 90:  # active this quarter
            score += 0.15
        elif days_inactive <= 180: # active in last 6 months
            score += 0.05
        else:                      # inactive 6+ months — big penalty
            score += 0.00
    except:
        score += 0.05

    # 3. Recruiter response rate — will they reply?
    response_rate = sig.get('recruiter_response_rate', 0)
    score += response_rate * 0.20  # max 0.20 points

    # 4. Notice period — JD wants sub-30 days
    notice = sig.get('notice_period_days', 90)
    if notice <= 15:
        score += 0.10  # immediately available
    elif notice <= 30:
        score += 0.08  # JD says this is ideal
    elif notice <= 60:
        score += 0.04  # ok but not ideal
    else:
        score += 0.01  # 90+ days = penalty

    # 5. Interview completion rate — do they ghost?
    icr = sig.get('interview_completion_rate', 0.5)
    score += icr * 0.10  # max 0.10 points

    # 6. Location match — Pune/Noida preferred
    location = candidate['profile'].get('location', '').lower()
    country = candidate['profile'].get('country', '').lower()
    willing_to_relocate = sig.get('willing_to_relocate', False)
    preferred_locations = ['pune', 'noida', 'delhi', 'gurgaon', 'gurugram', 'hyderabad', 'mumbai', 'bangalore', 'bengaluru']

    if any(loc in location for loc in preferred_locations):
        score += 0.05  # in a good Indian city
    elif 'india' in country and willing_to_relocate:
        score += 0.03  # India-based + willing to move

    # Clamp between 0.3 and 1.0
    # We never go to 0 — even inactive candidates might respond
    return max(0.3, min(score, 1.0))


def score_profile_quality(candidate):
    """
    Signal 5: How credible and complete is this candidate's profile?
    Returns a score from 0.0 to 1.0
    """
    sig = candidate.get('redrob_signals', {})
    score = 0.0

    # Profile completeness (0-100 scale → 0-0.3)
    completeness = sig.get('profile_completeness_score', 0)
    score += (completeness / 100) * 0.30

    # GitHub activity — JD explicitly cares about this
    github = sig.get('github_activity_score', -1)
    if github >= 0:  # -1 means no GitHub linked
        score += (github / 100) * 0.30
    # No GitHub = 0 bonus (not penalized, just no boost)

    # Skill assessment scores on Redrob platform — verified skill = more trust
    assessment_scores = sig.get('skill_assessment_scores', {})
    if assessment_scores:
        avg_assessment = sum(assessment_scores.values()) / len(assessment_scores)
        score += (avg_assessment / 100) * 0.20

    # Verified contact info — real person
    if sig.get('verified_email', False):
        score += 0.05
    if sig.get('verified_phone', False):
        score += 0.05

    # LinkedIn connected — cross-platform presence
    if sig.get('linkedin_connected', False):
        score += 0.05

    # Saved by recruiters — external validation of quality
    saved = sig.get('saved_by_recruiters_30d', 0)
    score += min(saved / 20.0, 0.05)  # max 0.05 bonus

    return min(score, 1.0)


def generate_reasoning(candidate, signals_breakdown):
    """
    Generate a human-readable explanation of WHY this candidate ranked here.
    This is our Innovation A — no other beginner will include this.
    """
    p = candidate['profile']
    sig = candidate.get('redrob_signals', {})

    title = p.get('current_title', 'Unknown')
    years = p.get('years_of_experience', 0)

    # Count AI skills
    candidate_skill_names = ' '.join([s['name'].lower() for s in candidate.get('skills', [])])
    ai_skill_count = sum(1 for skill in REQUIRED_SKILLS if skill.lower() in candidate_skill_names)

    response_rate = sig.get('recruiter_response_rate', 0)
    open_flag = 'open to work' if sig.get('open_to_work_flag') else 'not open to work'

    # Get last active info
    try:
        last_active = datetime.strptime(sig.get('last_active_date', '2020-01-01'), '%Y-%m-%d').date()
        days_inactive = (date.today() - last_active).days
        if days_inactive <= 30:
            active_str = 'active this month'
        elif days_inactive <= 90:
            active_str = f'active {days_inactive}d ago'
        else:
            active_str = f'inactive {days_inactive}d'
    except:
        active_str = 'unknown activity'

    # Semantic score description
    sem_score = signals_breakdown['semantic']
    if sem_score >= 0.7:
        sem_desc = 'strong semantic match'
    elif sem_score >= 0.5:
        sem_desc = 'moderate semantic match'
    else:
        sem_desc = 'weak semantic match'

    reasoning = (
        f"{title} | {years:.1f} yrs exp | {ai_skill_count} AI skills matched | "
        f"{sem_desc} | {open_flag} | {active_str} | "
        f"response rate {response_rate:.0%}"
    )
    return reasoning


print('✅ All scoring functions defined!')
print('   Functions ready: build_candidate_text, score_skill_match,')
print('   score_role_relevance, score_behavioral_availability,')
print('   score_profile_quality, generate_reasoning')

✅ All scoring functions defined!
   Functions ready: build_candidate_text, score_skill_match,
   score_role_relevance, score_behavioral_availability,
   score_profile_quality, generate_reasoning


## CELL 7 — Test on 50 sample candidates FIRST
⚠️ ALWAYS test on sample_candidates.json before running on the full 100K.
This runs in ~30 seconds. Make sure it works here before moving to Cell 9.

In [ ]:
# ── Load sample candidates ──
SAMPLE_PATH = os.path.join(DATASET_FOLDER, 'sample_candidates.json')

with open(SAMPLE_PATH, 'r') as f:
    sample_candidates = json.load(f)

print(f'Loaded {len(sample_candidates)} sample candidates')
print(f'First candidate: {sample_candidates[0]["candidate_id"]} — {sample_candidates[0]["profile"]["current_title"]}')

Loaded 50 sample candidates
First candidate: CAND_0000001 — Backend Engineer


In [ ]:
def rank_candidates(candidates, model, job_vector, top_n=100):
    """
    Main ranking function.
    Takes a list of candidates, returns a sorted dataframe with scores.
    """
    print(f'Processing {len(candidates)} candidates...')

    # ── Step 1: Build text for every candidate ──
    print('  Step 1/4: Building candidate text profiles...')
    candidate_texts = [build_candidate_text(c) for c in tqdm(candidates)]

    # ── Step 2: Convert all texts to vectors using the AI model ──
    print('  Step 2/4: Encoding with AI model (this is the slow step)...')
    # batch_size=64 means process 64 candidates at a time — uses memory efficiently
    candidate_vectors = model.encode(
        candidate_texts,
        batch_size=64,
        show_progress_bar=True
    )

    # ── Step 3: Calculate cosine similarity (Signal 1 — semantic match) ──
    print('  Step 3/4: Calculating semantic similarity scores...')
    # cosine_similarity returns a score 0-1 for each candidate
    semantic_scores = cosine_similarity(job_vector, candidate_vectors)[0]

    # ── Step 4: Score each candidate on all 5 signals ──
    print('  Step 4/4: Scoring all signals for each candidate...')
    results = []

    for i, candidate in enumerate(tqdm(candidates)):
        # Get each signal score
        semantic    = float(semantic_scores[i])          # Signal 1: AI meaning match
        skill       = score_skill_match(candidate, REQUIRED_SKILLS)       # Signal 2: keyword skills
        role        = score_role_relevance(candidate, GOOD_TITLES, CONSULTING_FIRMS)   # Signal 3: right job type
        profile_q   = score_profile_quality(candidate)   # Signal 5: credibility

        # Signal 4: behavioral availability used as a MULTIPLIER
        availability_multiplier = score_behavioral_availability(candidate)

        # ── FINAL SCORE FORMULA ──
        # Weighted combination of signals 1-3 and 5:
        raw_score = (
            semantic  * 0.45 +   # 45% — semantic understanding
            skill     * 0.25 +   # 25% — skill keyword match
            role      * 0.20 +   # 20% — right job role/career
            profile_q * 0.10     # 10% — profile quality
        )

        # Apply behavioral multiplier (0.3 to 1.0)
        # This ensures unavailable/inactive candidates rank lower
        final_score = raw_score * availability_multiplier

        # Generate the reasoning explanation
        signals_breakdown = {'semantic': semantic, 'skill': skill, 'role': role}
        reasoning = generate_reasoning(candidate, signals_breakdown)

        results.append({
            'candidate_id': candidate['candidate_id'],
            'final_score': round(final_score, 4),
            'semantic_score': round(semantic, 4),
            'skill_score': round(skill, 4),
            'role_score': round(role, 4),
            'availability_multiplier': round(availability_multiplier, 4),
            'profile_quality_score': round(profile_q, 4),
            'reasoning': reasoning,
        })

    # Convert to DataFrame and sort by final score
    df = pd.DataFrame(results)
    df = df.sort_values('final_score', ascending=False).reset_index(drop=True)

    print(f'\n✅ Ranking complete! Returning top {min(top_n, len(df))} candidates.')
    return df


# ── Run on sample candidates ──
sample_results = rank_candidates(sample_candidates, model, job_vector)

print('\n📊 TOP 10 RANKED CANDIDATES (sample):')
print(sample_results[['candidate_id','final_score','semantic_score','skill_score','role_score','availability_multiplier']].head(10).to_string())

Processing 50 candidates...
  Step 1/4: Building candidate text profiles...


100%|██████████| 50/50 [00:00<00:00, 55494.89it/s]

  Step 2/4: Encoding with AI model (this is the slow step)...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

  Step 3/4: Calculating semantic similarity scores...
  Step 4/4: Scoring all signals for each candidate...


100%|██████████| 50/50 [00:00<00:00, 3524.33it/s]


✅ Ranking complete! Returning top 50 candidates.

📊 TOP 10 RANKED CANDIDATES (sample):
   candidate_id  final_score  semantic_score  skill_score  role_score  availability_multiplier
0  CAND_0000031       0.6425          0.6688         1.00        0.60                    0.882
1  CAND_0000001       0.5366          0.5162         1.00        1.00                    0.729
2  CAND_0000027       0.3245          0.4973         0.60        0.45                    0.637
3  CAND_0000038       0.3111          0.5437         0.45        0.60                    0.605
4  CAND_0000015       0.3005          0.5378         0.50        1.00                    0.496
5  CAND_0000048       0.2865          0.4922         0.20        0.60                    0.682
6  CAND_0000025       0.2523          0.5342         0.25        0.10                    0.678
7  CAND_0000010       0.2436          0.5488         0.80        1.00                    0.343
8  CAND_0000014       0.2376          0.5803         0.30

In [ ]:
# ── Sanity check — look at the top 5 in detail ──
print('TOP 5 CANDIDATE DETAILS:\n')
for idx, row in sample_results.head(5).iterrows():
    cand_id = row['candidate_id']
    # Find original candidate data
    cand = next(c for c in sample_candidates if c['candidate_id'] == cand_id)
    p = cand['profile']
    sig = cand['redrob_signals']
    print(f"Rank {idx+1}: {cand_id}")
    print(f"  Title:          {p['current_title']}")
    print(f"  Experience:     {p['years_of_experience']} years")
    print(f"  Location:       {p['location']}, {p['country']}")
    print(f"  Open to work:   {sig['open_to_work_flag']}")
    print(f"  Last active:    {sig['last_active_date']}")
    print(f"  Response rate:  {sig['recruiter_response_rate']}")
    print(f"  Final score:    {row['final_score']}")
    print(f"  Reasoning:      {row['reasoning']}")
    print()

TOP 5 CANDIDATE DETAILS:

Rank 1: CAND_0000031
  Title:          Recommendation Systems Engineer
  Experience:     6.0 years
  Location:       Hyderabad, Telangana, India
  Open to work:   True
  Last active:    2026-05-24
  Response rate:  0.91
  Final score:    0.6425
  Reasoning:      Recommendation Systems Engineer | 6.0 yrs exp | 12 AI skills matched | moderate semantic match | open to work | active this month | response rate 91%

Rank 2: CAND_0000001
  Title:          Backend Engineer
  Experience:     6.9 years
  Location:       Toronto, Canada
  Open to work:   True
  Last active:    2026-05-20
  Response rate:  0.34
  Final score:    0.5366
  Reasoning:      Backend Engineer | 6.9 yrs exp | 6 AI skills matched | moderate semantic match | open to work | active this month | response rate 34%

Rank 3: CAND_0000027
  Title:          DevOps Engineer
  Experience:     3.9 years
  Location:       Kolkata, West Bengal, India
  Open to work:   True
  Last active:    2026-05-07
  Respon

## CELL 8 — Sanity check questions
Before running on the full dataset, ask yourself these 3 questions.
If the answer to all 3 is YES — proceed. If NO — come back and fix.

In [ ]:
print('=== SANITY CHECK ===')
print()

top5 = sample_results.head(5)
top5_ids = top5['candidate_id'].tolist()
top5_candidates = [c for c in sample_candidates if c['candidate_id'] in top5_ids]

# Q1: Are top candidates in engineering/AI roles?
titles = [c['profile']['current_title'] for c in top5_candidates]
print('Q1: Are top 5 in engineering/AI roles?')
for i, (cid, title) in enumerate(zip(top5_ids, titles)):
    print(f'   #{i+1} {cid}: {title}')

print()

# Q2: Are top candidates open to work or recently active?
print('Q2: Are top 5 reachable (open to work / recently active)?')
for i, c in enumerate(top5_candidates):
    sig = c['redrob_signals']
    print(f"   #{i+1}: open={sig['open_to_work_flag']}, last_active={sig['last_active_date']}, resp={sig['recruiter_response_rate']}")

print()

# Q3: Does the lowest-ranked candidate make sense?
last = sample_results.tail(1).iloc[0]
last_cand = next(c for c in sample_candidates if c['candidate_id'] == last['candidate_id'])
print('Q3: Does the lowest-ranked candidate make sense (should be a poor fit)?')
print(f"   Lowest: {last['candidate_id']} | {last_cand['profile']['current_title']} | score={last['final_score']}")
print(f"   Open: {last_cand['redrob_signals']['open_to_work_flag']} | last active: {last_cand['redrob_signals']['last_active_date']}")

print()
print('If the above looks reasonable — proceed to Cell 9 (full dataset run)!')
print('If something looks wrong — go back to Cell 6 and adjust the weights.')

=== SANITY CHECK ===

Q1: Are top 5 in engineering/AI roles?
   #1 CAND_0000031: Backend Engineer
   #2 CAND_0000001: Software Engineer
   #3 CAND_0000027: DevOps Engineer
   #4 CAND_0000038: Recommendation Systems Engineer
   #5 CAND_0000015: Java Developer

Q2: Are top 5 reachable (open to work / recently active)?
   #1: open=True, last_active=2026-05-20, resp=0.34
   #2: open=True, last_active=2026-02-12, resp=0.32
   #3: open=True, last_active=2026-05-07, resp=0.58
   #4: open=True, last_active=2026-05-24, resp=0.91
   #5: open=True, last_active=2026-03-31, resp=0.2

Q3: Does the lowest-ranked candidate make sense (should be a poor fit)?
   Lowest: CAND_0000017 | Accountant | score=0.0712
   Open: False | last active: 2025-11-04

If the above looks reasonable — proceed to Cell 9 (full dataset run)!
If something looks wrong — go back to Cell 6 and adjust the weights.


## CELL 9 — Run on full 100,000 candidates
⚠️ Only run this AFTER the sample test above looks correct.
This will take 20-40 minutes on Colab CPU. Go make tea ☕

**Tip:** Enable GPU in Colab to make this faster:
Runtime → Change runtime type → GPU → Save

In [ ]:
FULL_DATASET_PATH = os.path.join(DATASET_FOLDER, 'candidates.jsonl')

print('Loading full dataset (487 MB — takes 1-2 min to read)...')
all_candidates = []
with open(FULL_DATASET_PATH, 'r') as f:
    for line in tqdm(f):
        line = line.strip()
        if line:  # skip empty lines
            all_candidates.append(json.loads(line))

print(f'\n✅ Loaded {len(all_candidates):,} candidates!')

Loading full dataset (487 MB — takes 1-2 min to read)...


100000it [00:11, 8461.54it/s]


✅ Loaded 100,000 candidates!


In [ ]:
# Run ranking on ALL 100,000 candidates
# This is the big one — 20-40 minutes
print('Starting full ranking...')
print('Estimated time: 20-40 min on CPU, 5-10 min on GPU')
print('Do not close this tab!\n')

full_results = rank_candidates(all_candidates, model, job_vector, top_n=100)

print('\n🎉 FULL RANKING COMPLETE!')
print(f'Total candidates ranked: {len(full_results):,}')
print('\nTop 10:')
print(full_results[['candidate_id','final_score','reasoning']].head(10).to_string())

Starting full ranking...
Estimated time: 20-40 min on CPU, 5-10 min on GPU
Do not close this tab!

Processing 100000 candidates...
  Step 1/4: Building candidate text profiles...


100%|██████████| 100000/100000 [00:00<00:00, 120514.27it/s]

  Step 2/4: Encoding with AI model (this is the slow step)...


Batches:   0%|          | 0/1563 [00:00<?, ?it/s]

  Step 3/4: Calculating semantic similarity scores...
  Step 4/4: Scoring all signals for each candidate...


100%|██████████| 100000/100000 [00:17<00:00, 5637.64it/s]



✅ Ranking complete! Returning top 100 candidates.

🎉 FULL RANKING COMPLETE!
Total candidates ranked: 100,000

Top 10:
   candidate_id  final_score                                                                                                                                             reasoning
0  CAND_0046132       0.7876               AI Research Engineer | 4.3 yrs exp | 7 AI skills matched | strong semantic match | open to work | active this month | response rate 94%
1  CAND_0018499       0.7825  Senior Machine Learning Engineer | 7.2 yrs exp | 10 AI skills matched | strong semantic match | open to work | active this month | response rate 61%
2  CAND_0039754       0.7767         Senior Applied Scientist | 16.2 yrs exp | 13 AI skills matched | strong semantic match | open to work | active this month | response rate 81%
3  CAND_0011687       0.7662               Senior NLP Engineer | 7.8 yrs exp | 10 AI skills matched | strong semantic match | open to work | active this month | resp

## CELL 10 — Generate submission CSV
Creates the exact format required by the challenge.

In [ ]:
# Take only the top 100 candidates
top_100 = full_results.head(100).copy()

# Add rank column (1 to 100)
top_100['rank'] = range(1, 101)

# Rename score column to match required format
top_100 = top_100.rename(columns={'final_score': 'score'})

# Select only the 4 required columns in the exact order
submission = top_100[['candidate_id', 'rank', 'score', 'reasoning']]

# ⚠️ IMPORTANT: Make sure scores are non-increasing
# (each rank must have score >= next rank's score)
scores = submission['score'].tolist()
for i in range(len(scores)-1):
    if scores[i] < scores[i+1]:
        print(f'WARNING: Score issue at rank {i+1} and {i+2}!')
        # Fix: small epsilon to maintain order
        scores[i+1] = scores[i] - 0.0001
submission['score'] = [round(s, 4) for s in scores]

# Save to file
OUTPUT_PATH = os.path.join(DATASET_FOLDER, 'submission.csv')
submission.to_csv(OUTPUT_PATH, index=False)

print(f'✅ Submission saved to: {OUTPUT_PATH}')
print(f'Rows: {len(submission)} (must be exactly 100)')
print(f'Score range: {submission["score"].max():.4f} to {submission["score"].min():.4f}')
print()
print('Preview:')
print(submission.head(5).to_string())

✅ Submission saved to: /content/drive/MyDrive/hackathon_data/submission.csv
Rows: 100 (must be exactly 100)
Score range: 0.7876 to 0.6593

Preview:
   candidate_id  rank   score                                                                                                                                             reasoning
0  CAND_0046132     1  0.7876               AI Research Engineer | 4.3 yrs exp | 7 AI skills matched | strong semantic match | open to work | active this month | response rate 94%
1  CAND_0018499     2  0.7825  Senior Machine Learning Engineer | 7.2 yrs exp | 10 AI skills matched | strong semantic match | open to work | active this month | response rate 61%
2  CAND_0039754     3  0.7767         Senior Applied Scientist | 16.2 yrs exp | 13 AI skills matched | strong semantic match | open to work | active this month | response rate 81%
3  CAND_0011687     4  0.7662               Senior NLP Engineer | 7.8 yrs exp | 10 AI skills matched | strong semantic match | open 

/tmp/ipykernel_949/3252984497.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  submission['score'] = [round(s, 4) for s in scores]


In [ ]:
# ── FIX: Handle tie-breaking by candidate_id ascending ──
# When scores are equal, sort by candidate_id (ascending = smaller number first)
# This is required by the official submission rules

submission = submission.copy()

# Sort by score descending, then candidate_id ascending for ties
submission = submission.sort_values(
    by=['score', 'candidate_id'],
    ascending=[False, True]   # score: high→low, id: small→large
).reset_index(drop=True)

# Re-assign ranks 1–100 after re-sorting
submission['rank'] = range(1, 101)

# Re-save the fixed file
submission.to_csv(OUTPUT_PATH, index=False)

print('✅ Tie-breaking fixed!')
print('Preview of ranks 90-100:')
print(submission.iloc[89:][['candidate_id','rank','score']].to_string())

✅ Tie-breaking fixed!
Preview of ranks 90-100:
    candidate_id  rank   score
89  CAND_0070333    90  0.6654
90  CAND_0012840    91  0.6624
91  CAND_0083852    92  0.6624
92  CAND_0011643    93  0.6618
93  CAND_0005509    94  0.6609
94  CAND_0023215    95  0.6607
95  CAND_0038206    96  0.6607
96  CAND_0090300    97  0.6607
97  CAND_0033445    98  0.6605
98  CAND_0052744    99  0.6605
99  CAND_0003575   100  0.6593


## CELL 11 — Validate your submission
Run the official validator. If it says 'Submission is valid.' you're done!

In [ ]:
# Copy the validate_submission.py from the dataset folder
import shutil
validator_src = os.path.join(DATASET_FOLDER, 'validate_submission.py')
shutil.copy(validator_src, '/content/validate_submission.py')

# Run the official validator
!python /content/validate_submission.py {OUTPUT_PATH}

Submission is valid.


## CELL 12 — Download your submission
Download the CSV file to your laptop so you can submit it to the portal.

In [ ]:
from google.colab import files

# This triggers a download to your laptop
files.download(OUTPUT_PATH)

print('📥 Download started!')
print('Check your Downloads folder for submission.csv')
print()
print('Next steps:')
print('1. Rename the file to your team_id.csv (check the portal for your team ID)')
print('2. Upload to hack2skill.com portal')
print('3. Fill in submission_metadata.yaml and push to GitHub')
print('4. Share your GitHub link in the portal')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 Download started!
Check your Downloads folder for submission.csv

Next steps:
1. Rename the file to your team_id.csv (check the portal for your team ID)
2. Upload to hack2skill.com portal
3. Fill in submission_metadata.yaml and push to GitHub
4. Share your GitHub link in the portal


## CELL 13 — Score breakdown analysis (for your README)
Run this to understand your results and write your methodology section.

In [ ]:
print('=== SUBMISSION ANALYSIS (use this in your README) ===\n')

top_100_full = full_results.head(100)

print(f'Score statistics:')
print(f'  Highest score: {top_100_full["semantic_score"].max():.4f} (semantic)')
print(f'  Avg semantic score (top 100): {top_100_full["semantic_score"].mean():.4f}')
print(f'  Avg skill score (top 100):    {top_100_full["skill_score"].mean():.4f}')
print(f'  Avg role score (top 100):     {top_100_full["role_score"].mean():.4f}')
print(f'  Avg availability (top 100):   {top_100_full["availability_multiplier"].mean():.4f}')
print()

# Lookup titles for top 100
top_ids = set(top_100_full['candidate_id'].tolist())
top_cands = [c for c in all_candidates if c['candidate_id'] in top_ids]

from collections import Counter
titles = [c['profile']['current_title'] for c in top_cands]
print('Top job titles in final top-100:')
for title, count in Counter(titles).most_common(10):
    print(f'  {title}: {count}')

print()
open_count = sum(1 for c in top_cands if c['redrob_signals']['open_to_work_flag'])
print(f'Open to work: {open_count}/100 ({open_count}%)')

print()
print('Signal weight breakdown used:')
print('  Semantic similarity:    45%')
print('  Skill match:            25%')
print('  Role relevance:         20%')
print('  Profile quality:        10%')
print('  (Behavioral signals applied as multiplier 0.3–1.0)')

=== SUBMISSION ANALYSIS (use this in your README) ===

Score statistics:
  Highest score: 0.7863 (semantic)
  Avg semantic score (top 100): 0.6898
  Avg skill score (top 100):    0.9890
  Avg role score (top 100):     0.9690
  Avg availability (top 100):   0.8537

Top job titles in final top-100:
  AI Research Engineer: 16
  ML Engineer: 15
  Junior ML Engineer: 13
  Data Scientist: 11
  Senior Software Engineer (ML): 11
  Machine Learning Engineer: 5
  Senior Data Scientist: 5
  AI Engineer: 5
  Senior Machine Learning Engineer: 3
  NLP Engineer: 3

Open to work: 100/100 (100%)

Signal weight breakdown used:
  Semantic similarity:    45%
  Skill match:            25%
  Role relevance:         20%
  Profile quality:        10%
  (Behavioral signals applied as multiplier 0.3–1.0)
